In [4]:
import os
import pickle
import sys

import torch

root_dir = os.path.abspath("..")
package_dir = os.path.abspath(os.path.join(root_dir, "current_setpoints"))
optimization_dir = os.path.abspath(os.path.join(package_dir, "optimization"))
model_dir = os.path.abspath(os.path.join(package_dir, "model"))
data_dir = os.path.join(root_dir, "data")

os.makedirs(data_dir, exist_ok=True)

for path in [root_dir, package_dir, optimization_dir, model_dir]:
    if path not in sys.path:
        sys.path.append(path)

from current_setpoints.optimization import (
    ModelNeural,
    MotorOptimizer,
    calculate_grid,
    get_correction_grid,
    grid_to_data,
)
from current_setpoints.utils.neural import load_neural_model
from current_setpoints.utils.plotting import plot_grid_segments
from current_setpoints.data import IEEEMachine2, FluxValues

import numpy as np

def save_new_state():
    flux_registry = FluxValues()
    machine = IEEEMachine2(flux_registry)
    
    np.savez('state_new.npz',
             n_phases=machine.n_phases,
             n_ppairs=machine.n_ppairs,
             mat_crossc=machine.mat_crossc,
             R_stat=machine.R_stat,
             L_stat=machine.L_stat,
             mat_A=machine.mat_A,
             vec_b=machine.vec_b,
             flux_volt=machine.flux_volt,
             flux_torq=machine.flux_torq)
    
    print("✅ New state saved to state_new.npz")

if __name__ == "__main__":
    save_new_state()

✅ New state saved to state_new.npz


In [5]:
import sys
import os
import numpy as np

# Inject path for Jupyter notebook
sys.path.insert(0, os.path.abspath('..'))

from current_setpoints.data import IEEEMachine2, FluxValues
from current_setpoints.model import Transform  # Make sure this matches your filename

def save_new_transform():
    flux_registry = FluxValues()
    machine = IEEEMachine2(flux_registry)
    
    # Exact same test conditions
    omega = 314.15 
    add_volt_0 = True
    test_current_dq = np.array([10.0, -15.0, 5.0, 2.0])
    
    # Initialize
    trans = Transform(machine, omega, add_volt_0)
    
    # Calculate outputs
    curr_ph = trans.get_curr_ph(test_current_dq)
    volt_dq = trans.get_volt_dq(test_current_dq)
    volt_ph, volt_0, volt_raw = trans.get_volt_ph(test_current_dq)
    
    np.savez('trans_new.npz',
             mat_dq_to_ph=trans.mat_dq_to_ph,
             mat_curr_dq_to_volt_dq_fixed=trans.mat_curr_dq_to_volt_dq_fixed,
             mat_curr_dq_to_volt_dq_omega=trans.mat_curr_dq_to_volt_dq_omega,
             curr_ph=curr_ph,
             volt_dq=volt_dq,
             volt_ph=volt_ph,
             volt_0=volt_0,
             volt_raw=volt_raw)
    
    print("✅ New transform outputs saved to trans_new.npz")

save_new_transform()

✅ New transform outputs saved to trans_new.npz


In [2]:
import sys
import os
import numpy as np

sys.path.insert(0, os.path.abspath('..'))

from current_setpoints.data import IEEEMachine2, FluxValues
from current_setpoints.model import Transform
from current_setpoints.optimization import ModelAnalytical, MotorOptimizer


def save_new_opt():
    flux_registry = FluxValues()
    machine = IEEEMachine2(flux_registry)
    
    # FIX: Inject the real physical limits (overriding the 0.0 Pylance fallbacks)
    machine.curr_max = 20.0
    machine.volt_max = 200.0
    # machine.speed_max = 3000.0 # (If you also use a speed limit)
    
    omega = 314.15
    W = Transform(machine, omega, True)
    
    # Initialize the new decoupled architecture
    model = ModelAnalytical(machine)
    optimizer = MotorOptimizer(model)
    
    print("Running New Max Torque Optimization...")
    mt_curr, mt_torq, mt_succ = optimizer.maximize_torque(W)
    
    print("Running New Target Torque Optimization (Target = 5.0 Nm)...")
    target_torq = 5.0
    dt_curr, dt_succ = optimizer.minimize_current(target_torq, W)
    
    np.savez('opt_new.npz',
             mt_curr=mt_curr,
             mt_torq=mt_torq,
             mt_succ=mt_succ,
             dt_curr=dt_curr,
             dt_succ=dt_succ)
    
    print("✅ New optimization outputs saved to opt_new.npz")

if __name__ == "__main__":
    save_new_opt()

Running New Max Torque Optimization...
Running New Target Torque Optimization (Target = 5.0 Nm)...
✅ New optimization outputs saved to opt_new.npz


In [3]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
import os
import sys

# IMPORTANT: Ensure your new `NeuralTorquePredictor` and `torq_analytical` 
# are imported here, or defined above this line in your actual file.

root_dir = os.path.abspath("..")
package_dir = os.path.abspath(os.path.join(root_dir, "current_setpoints"))
optimization_dir = os.path.abspath(os.path.join(package_dir, "optimization"))
model_dir = os.path.abspath(os.path.join(package_dir, "model"))
data_dir = os.path.join(root_dir, "data")

os.makedirs(data_dir, exist_ok=True)

for path in [root_dir, package_dir, optimization_dir, model_dir]:
    if path not in sys.path:
        sys.path.append(path)

from current_setpoints.optimization import (
    ModelNeural,
    MotorOptimizer,
    calculate_grid,
    get_correction_grid,
    grid_to_data,
)
from current_setpoints.utils.neural import NeuralTorquePredictor, torq_analytical


class DummyMachineNew:
    def __init__(self):
        self.mat_A = np.eye(4)
        self.vec_b = np.ones((4,))
        
    def update_state(self, omega, vec_curr_dq):
        pass # Mock update

def main():
    device = torch.device("cpu")
    hidden_size = 16

    # 1. Load the identical inputs generated by the old model script
    try:
        data = np.load("shared_data.npz")
        x_normed = torch.from_numpy(data["x_normed"]).float().to(device)
    except FileNotFoundError:
        print("Error: 'shared_data.npz' not found. Please run run_old_model.py first and copy the file here.")
        return

    batch_size, input_size = x_normed.shape

    # 2. Init Machine and Scaler
    dummy_scaler = StandardScaler()
    dummy_scaler.mean_ = np.zeros(input_size)
    dummy_scaler.scale_ = np.ones(input_size)
    machine_new = DummyMachineNew()

    # 3. Init Model
    new_model = NeuralTorquePredictor(input_size, hidden_size, dummy_scaler, machine_new, device)
    new_model.eval()
    
    # 4. Format Dynamic B_Tensor [batch, features, 1]
    B_tensor_dynamic = torch.from_numpy(machine_new.vec_b).float().to(device).unsqueeze(0).unsqueeze(-1)
    B_tensor_dynamic = B_tensor_dynamic.expand(batch_size, -1, -1) 

    # 5. Predict
    with torch.no_grad():
        out_new = new_model(x_normed, B_tensor_dynamic)

    # 6. Save new results
    np.savez("new_results.npz", out_new=out_new.numpy())
    print("New model run complete. Saved 'new_results.npz'.")

if __name__ == "__main__":
    main()

New model run complete. Saved 'new_results.npz'.


In [4]:
import numpy as np

def main():
    try:
        old_data = np.load("shared_data.npz")
        new_data = np.load("new_results.npz")
        
        out_old = old_data["out_old"]
        out_new = new_data["out_new"]

        print("--- Output Shapes ---")
        print(f"Old Model: {out_old.shape}")
        print(f"New Model: {out_new.shape}")

        print("\n--- Value Comparison ---")
        print("Old Model Outputs:\n", out_old)
        print("\nNew Model Outputs:\n", out_new)

        diff = np.abs(out_old - out_new)
        print("\n--- Absolute Difference ---")
        print(diff)

        # Using atol=1e-5 to account for minor floating point differences 
        # that might occur between different environment/library versions.
        is_close = np.allclose(out_old, out_new, atol=1e-5)
        
        print("\n==============================================")
        if is_close:
            print("RESULT: PASS. Outputs are mathematically equivalent.")
        else:
            print("RESULT: FAIL. Outputs are significantly different.")
            print("Note: If the outputs differ, recall that the new model multiplies the linear torque by 2!")
        print("==============================================")

    except FileNotFoundError as e:
        print(f"Error loading files: {e}")
        print("Ensure both 'shared_data.npz' and 'new_results.npz' are in this directory.")

if __name__ == "__main__":
    main()

--- Output Shapes ---
Old Model: (3, 1)
New Model: (3, 1)

--- Value Comparison ---
Old Model Outputs:
 [[0.8220746]
 [7.998155 ]
 [4.835179 ]]

New Model Outputs:
 [[1.1497526]
 [8.33853  ]
 [5.601828 ]]

--- Absolute Difference ---
[[0.32767802]
 [0.34037447]
 [0.76664925]]

RESULT: FAIL. Outputs are significantly different.
Note: If the outputs differ, recall that the new model multiplies the linear torque by 2!


In [5]:
import torch
import os
import sys

# IMPORT YOUR NEW FUNCTION HERE
# from your_new_lib.neural_file import torq_analytical
root_dir = os.path.abspath("..")
package_dir = os.path.abspath(os.path.join(root_dir, "current_setpoints"))
optimization_dir = os.path.abspath(os.path.join(package_dir, "optimization"))
model_dir = os.path.abspath(os.path.join(package_dir, "model"))
data_dir = os.path.join(root_dir, "data")

os.makedirs(data_dir, exist_ok=True)

for path in [root_dir, package_dir, optimization_dir, model_dir]:
    if path not in sys.path:
        sys.path.append(path)

from current_setpoints.optimization import (
    ModelNeural,
    MotorOptimizer,
    calculate_grid,
    get_correction_grid,
    grid_to_data,
)
from current_setpoints.utils.neural import NeuralTorquePredictor, torq_analytical

def test_new():
    # Exact same dummy data
    currents = torch.tensor([[1.0, 2.0, 3.0, 4.0]]) # [batch, features]
    B_tensor = torch.tensor([[[0.1], [0.2], [0.3], [0.4]]]) # [batch, features, 1]
    A_tensor = torch.eye(4) # [features, features]

    # Run the new analytical math
    out_new = torq_analytical(currents, A_tensor, B_tensor)
    print(f"NEW Model Analytical Output: {out_new.item()}")

if __name__ == "__main__":
    test_new()

NEW Model Analytical Output: 33.0
